# 🏥 Praxirence Clinical AI Training Pipeline (Google Colab T4 GPU)

This notebook orchestrates the complete end-to-end training of:
1. **Whisper ASR Model** fine-tuned via **LoRA (PEFT)** for medical consultation speech recognition.
2. **Mistral-7B / Llama-3-8B Care-Plan LLM** fine-tuned via **QLoRA (4-bit quantization)** to extract diagnosis, prescribed medications, and reminder schedules into structured JSON.

### Hardware Requirements:
- Google Colab Free Tier with **T4 GPU (16GB VRAM)**.
- Total training time: ~3 to 4 hours.

In [73]:
# Step 1: Verify Google Colab GPU
!nvidia-smi

Thu Sep  3 18:10:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [74]:
# Step 2: Mount Google Drive to persist trained adapters
from google.colab import drive
drive.mount('/content/drive')

# Create destination directory on Google Drive
!mkdir -p /content/drive/MyDrive/praxirence_models/asr_adapter
!mkdir -p /content/drive/MyDrive/praxirence_models/careplan_adapter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [75]:
# Step 3: Clone your real repo and install dependencies
%cd /content
!rm -rf /content/Praxirence
!git clone https://github.com/M20A03/Praxirence.git
%cd /content/Praxirence
!git pull
%cd /content/Praxirence/ml_pipeline

!pip install --upgrade pip
!pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.28.0 trl>=0.8.0 datasets librosa soundfile jiwer evaluate rouge-score sacrebleu

/content
Cloning into 'Praxirence'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (290/290), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 290 (delta 61), reused 279 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (290/290), 653.85 KiB | 1.38 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/Praxirence
Already up to date.
/content/Praxirence/ml_pipeline
Looking in indexes: https://download.pytorch.org/whl/cu121


In [76]:
# Step 4: Programmatically fetch and prepare free medical speech datasets
!python scripts/data_fetch.py --dataset tobiolatunji/afrispeech-200

2026-09-03 18:10:16,585 [INFO] Target raw directory: /content/Praxirence/ml_pipeline/scripts/../data/raw
2026-09-03 18:10:16,586 [INFO] Data fetch complete! Stored 5 samples with manifest at: /content/Praxirence/ml_pipeline/scripts/../data/raw/metadata.json


In [77]:
# Step 5: Preprocess audio (Resample to 16kHz mono, VAD silence trim, peak volume normalization)
!python scripts/preprocess_audio.py

2026-09-03 18:10:16,737 [INFO] Preprocessed audio: consult_001.wav -> 16kHz Mono VAD Normalized
2026-09-03 18:10:16,775 [INFO] Preprocessed audio: consult_002.wav -> 16kHz Mono VAD Normalized
2026-09-03 18:10:16,814 [INFO] Preprocessed audio: consult_003.wav -> 16kHz Mono VAD Normalized
2026-09-03 18:10:16,855 [INFO] Preprocessed audio: consult_004.wav -> 16kHz Mono VAD Normalized
2026-09-03 18:10:16,893 [INFO] Preprocessed audio: consult_005.wav -> 16kHz Mono VAD Normalized
2026-09-03 18:10:16,894 [INFO] Audio preprocessing complete! Saved 5 processed files to /content/Praxirence/ml_pipeline/scripts/../data/processed/metadata.json


In [78]:
# Step 6: Preprocess text into instruction-tuning JSONL pairs (train.jsonl & val.jsonl)
!python scripts/preprocess_text.py --val_ratio 0.2

2026-09-03 18:10:17,017 [INFO] Generated instruction datasets successfully:
2026-09-03 18:10:17,017 [INFO]  - Train: 4 samples -> /content/Praxirence/ml_pipeline/scripts/../dataset/train.jsonl
2026-09-03 18:10:17,017 [INFO]  - Val:   1 samples -> /content/Praxirence/ml_pipeline/scripts/../dataset/val.jsonl


In [79]:
# Step 7: Fine-tune Whisper ASR using LoRA
!pip uninstall -y torchao
!python scripts/train_asr.py --base_model openai/whisper-small --batch_size 4 --epochs 3 --output_dir models/asr_adapter

2026-09-03 18:10:20,942 [INFO] Initializing ASR Training on device: CUDA (FP16: True)
2026-09-03 18:10:29,024 [INFO] NumExpr defaulting to 2 threads.
2026-09-03 18:10:31,160 [INFO] TensorFlow version 2.20.0 available.
2026-09-03 18:10:31,161 [INFO] JAX version 0.11.1 available.
2026-09-03 18:10:31,464 [INFO] Loading Whisper processor and base model: openai/whisper-small...
2026-09-03 18:10:31,686 [INFO] HTTP Request: GET https://huggingface.co/api/models/openai/whisper-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-09-03 18:10:31,850 [INFO] HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-03 18:10:31,974 [INFO] HTTP Request: HEAD https://huggingface.co/openai/whisper-small/resolve/main/chat_template.json "HTTP/1.1 404 Not Found"
2026-09-03 18:10:31,974 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to e

In [95]:
# Step 8: Fine-tune Care-Plan LLM using QLoRA 4-bit
!python scripts/train_llm.py \
    --base_model mistralai/Mistral-7B-Instruct-v0.2 \
    --batch_size 1 \
    --grad_accum 4 \
    --epochs 3 \
    --output_dir models/careplan_adapter

2026-09-03 18:44:29,655 [INFO] NumExpr defaulting to 2 threads.
2026-09-03 18:44:31,979 [INFO] TensorFlow version 2.20.0 available.
2026-09-03 18:44:31,980 [INFO] JAX version 0.11.1 available.
2026-09-03 18:44:32,445 [INFO] Loading tokenizer: mistralai/Mistral-7B-Instruct-v0.2...
2026-09-03 18:44:32,639 [INFO] HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-03 18:44:32,640 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-03 18:44:32,649 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mistralai/Mistral-7B-Instruct-v0.2/63a8b081895390a26e140280378bc85ec8bce07a/config.json "HTTP/1.1 200 OK"
2026-09-03 18:44:32,761 [INFO] HTTP Request: HEAD https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"

In [96]:
# Step 9: Evaluate both models (WER, CER, ROUGE-L, BLEU) and generate HTML report
import os

# 1. Ensure we are in the ml_pipeline directory
if os.path.exists("/content/Praxirence/ml_pipeline"):
    os.chdir("/content/Praxirence/ml_pipeline")

# 2. Run the evaluation script
!python scripts/evaluate.py

# 3. Locate the generated evaluation HTML report
report_path = "evaluation_report.html"
if not os.path.exists(report_path) and os.path.exists("/content/Praxirence/ml_pipeline/evaluation_report.html"):
    report_path = "/content/Praxirence/ml_pipeline/evaluation_report.html"

# 4. Display the scorecard directly inside the Colab notebook
from IPython.display import HTML, display
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        display(HTML(f.read()))
else:
    print("✅ Evaluation completed successfully! Metrics logged above.")

2026-09-03 18:47:47,457 [INFO] NumExpr defaulting to 2 threads.
2026-09-03 18:47:48,077 [INFO] Using default tokenizer.
2026-09-03 18:47:48,191 [INFO] Evaluation HTML report generated at: /content/Praxirence/ml_pipeline/scripts/../evaluation_report.html
2026-09-03 18:47:48,191 [INFO] Evaluation complete! Metrics:
{
  "asr": {
    "wer": 0.0,
    "cer": 0.0
  },
  "llm": {
    "rouge1": 1.0,
    "rouge2": 1.0,
    "rougeL": 1.0,
    "bleu": 1.0
  },
  "samples_evaluated": 1
}


In [87]:
# Step 10: Copy fine-tuned LoRA & QLoRA adapters to Google Drive for permanent backup
!cp -r models/asr_adapter /content/drive/MyDrive/praxirence_models/
!cp -r models/careplan_adapter /content/drive/MyDrive/praxirence_models/
print("Adapters safely saved to Google Drive!")

Adapters safely saved to Google Drive!


In [86]:
%cd /content/Praxirence
!git pull
%cd /content/Praxirence/ml_pipeline

/content/Praxirence
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 571 bytes | 571.00 KiB/s, done.
From https://github.com/M20A03/Praxirence
   b4ebb8c..e9fd2c6  main       -> origin/main
Updating b4ebb8c..e9fd2c6
Fast-forward
 ml_pipeline/scripts/train_llm.py | 15 ++++++---------
 1 file changed, 6 insertions(+), 9 deletions(-)
/content/Praxirence/ml_pipeline
